# Version 2 — Model Complexity Experiments

## Objective

In this experiment, I investigate whether increasing model complexity
improves the accuracy of my Cristiano Ronaldo goal-count forecasting models.

My original ensemble combines XGBoost and CatBoost Poisson regressors.
Although the ensemble performs slightly better than the individual models
during chronological validation, its predictions are relatively conservative.

I want to determine whether this behavior reflects excessive regularization
or genuinely limited predictive information in the available features.

## Experimental approach

I will:

1. Load the original engineered dataset and frozen modeling contracts.
2. Reproduce the selected Version 1 models using chronological validation.
3. Test controlled changes to model complexity.
4. Compare candidates using macro Poisson deviance and supporting metrics.
5. Examine fold stability before selecting any improvement.

## Evaluation principles

- I will preserve the original four expanding-window validation folds.
- I will fit preprocessing separately within each training fold.
- I will use macro Poisson deviance as my primary selection metric.
- I will compare results against the original ensemble benchmark of 1.106496.
- I will not use the original final-test observations for model selection.
- I will keep all Version 2 experiments separate from the original project.

The goal is improved predictive accuracy, not simply more variable predictions.

In [2]:
# Load the original dataset and frozen modeling contracts

import json
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve paths from the Version 2 notebook directory
PROJECT_ROOT = Path.cwd().resolve().parents[2]

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "engineered"
    / "ronaldo_1000_ml_engineered.csv"
)

ARTIFACT_DIR = PROJECT_ROOT / "artifacts"

# Load historical data
df = pd.read_csv(
    DATA_PATH,
    parse_dates=["date"]
)

df = df.sort_values("date").reset_index(drop=True)

# Load original modeling contracts
contract_files = {
    "feature_spec": "feature_spec.json",
    "temporal_folds": "temporal_folds.json",
    "validation_policy": "validation_policy.json",
    "preprocessing_spec": "preprocessing_spec.json",
}

contracts = {}

for name, filename in contract_files.items():
    with open(ARTIFACT_DIR / filename, encoding="utf-8") as f:
        contracts[name] = json.load(f)

# Restore original definitions
TARGET = contracts["feature_spec"]["primary_target"]

FEATURE_SETS = contracts["preprocessing_spec"]["feature_sets"]

NUMERIC_FEATURES = contracts["feature_spec"]["numeric_features"]

CATEGORICAL_FEATURES = contracts["feature_spec"]["categorical_features"]

TEMPORAL_FOLDS = contracts["temporal_folds"]

FINAL_TEST_START = pd.Timestamp(
    contracts["validation_policy"]["final_test_start"]
)

# Keep final-test observations outside development
development_df = df[df["date"] < FINAL_TEST_START].copy()

print("VERSION 2 — DATA AND CONTRACT LOADING")
print("=" * 55)

print("Project root:", PROJECT_ROOT)
print("Dataset shape:", df.shape)
print("Development appearances:", len(development_df))
print("Target:", TARGET)
print("Feature sets:", {k: len(v) for k, v in FEATURE_SETS.items()})
print("Temporal folds:", len(TEMPORAL_FOLDS))
print("Final-test boundary:", FINAL_TEST_START.date())

# Integrity checks
assert len(df) == 1333
assert len(development_df) == 1255
assert len(TEMPORAL_FOLDS) == 4
assert TARGET == "target_goals"

print("\nAll initial integrity checks passed.")

VERSION 2 — DATA AND CONTRACT LOADING
Project root: C:\Users\JONATHAN\Desktop\WORK\ronaldo-1000-ml
Dataset shape: (1333, 100)
Development appearances: 1255
Target: target_goals
Feature sets: {'full': 84, 'context': 80, 'no_season': 83}
Temporal folds: 4
Final-test boundary: 2025-01-01

All initial integrity checks passed.


In [4]:
# Reconstruct and validate the original temporal folds

temporal_folds = []
validation_ids = []

for fold_spec in TEMPORAL_FOLDS:

    train_start = pd.Timestamp(fold_spec["train_start"])
    train_end = pd.Timestamp(fold_spec["train_end"])

    val_start = pd.Timestamp(fold_spec["validation_start"])
    val_end = pd.Timestamp(fold_spec["validation_end"])

    train = df.loc[
        df["date"].between(train_start, train_end)
    ].copy()

    val = df.loc[
        df["date"].between(val_start, val_end)
    ].copy()

    # Preserve chronological separation
    assert train["date"].max() < val["date"].min()

    # Keep final-test observations outside validation
    assert val["date"].max() < FINAL_TEST_START

    # Confirm saved fold sizes
    assert len(train) == fold_spec["train_matches"]
    assert len(val) == fold_spec["validation_matches"]

    temporal_folds.append({
        "name": fold_spec["name"],
        "train": train,
        "val": val,
    })

    validation_ids.extend(val["match_id"].tolist())

# Verify that validation observations are unique
assert len(validation_ids) == len(set(validation_ids))

# Verify original validation coverage
assert len(validation_ids) == 340

# Verify no overlap with final test
final_test_ids = set(
    df.loc[
        df["date"] >= FINAL_TEST_START,
        "match_id"
    ]
)

assert set(validation_ids).isdisjoint(final_test_ids)

print("VERSION 2 — TEMPORAL VALIDATION")
print("=" * 75)

for fold in temporal_folds:
    train = fold["train"]
    val = fold["val"]

    print(
        f"{fold['name']}: "
        f"Train {len(train):4d} matches | "
        f"Validation {len(val):3d} matches | "
        f"{val['date'].min().date()} → "
        f"{val['date'].max().date()}"
    )

print("\nTotal validation appearances:", len(validation_ids))
print("Chronological validation integrity: PASSED")

VERSION 2 — TEMPORAL VALIDATION
Fold 1: Train  915 matches | Validation 158 matches | 2018-08-18 → 2021-06-27
Fold 2: Train 1073 matches | Validation  72 matches | 2021-08-22 → 2022-12-10
Fold 3: Train 1145 matches | Validation  59 matches | 2023-01-22 → 2023-12-30
Fold 4: Train 1204 matches | Validation  51 matches | 2024-02-14 → 2024-12-06

Total validation appearances: 340
Chronological validation integrity: PASSED


In [6]:
# Define reusable goal-count evaluation functions

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_poisson_deviance,
)


def evaluate_goal_predictions(y_true, y_pred):
    """
    Evaluate goal-count predictions using the original
    Version 1 metric definitions.
    """

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    # Poisson predictions must remain strictly positive
    y_pred = np.clip(y_pred, 1e-9, None)

    return {
        "poisson_deviance": mean_poisson_deviance(
            y_true, y_pred
        ),
        "mae": mean_absolute_error(
            y_true, y_pred
        ),
        "rmse": np.sqrt(
            mean_squared_error(y_true, y_pred)
        ),
        "mean_bias": y_pred.mean() - y_true.mean(),
        "actual_mean_goals": y_true.mean(),
        "predicted_mean_goals": y_pred.mean(),
    }


def summarize_oof_predictions(oof):
    """
    Summarize chronological out-of-fold predictions.

    Expected columns:
        fold, y_true, y_pred
    """

    fold_results = []

    for fold_name, group in oof.groupby(
        "fold", sort=False
    ):
        metrics = evaluate_goal_predictions(
            group["y_true"],
            group["y_pred"],
        )

        fold_results.append({
            "fold": fold_name,
            "matches": len(group),
            **metrics,
        })

    fold_results = pd.DataFrame(fold_results)

    # Match the original macro-averaging approach
    metric_columns = [
        "poisson_deviance",
        "mae",
        "rmse",
        "mean_bias",
    ]

    macro_metrics = (
        fold_results[metric_columns]
        .mean()
        .to_dict()
    )

    pooled_metrics = evaluate_goal_predictions(
        oof["y_true"],
        oof["y_pred"],
    )

    return {
        "fold_results": fold_results,
        "macro_metrics": macro_metrics,
        "pooled_metrics": pooled_metrics,
    }


print("VERSION 2 — EVALUATION FUNCTIONS")
print("=" * 55)

# Simple sanity check
test_actual = np.array([0, 1, 2, 0])
test_predicted = np.array([0.4, 0.8, 1.3, 0.6])

test_metrics = evaluate_goal_predictions(
    test_actual,
    test_predicted,
)

for metric, value in test_metrics.items():
    print(f"{metric:<25}: {value:.6f}")

assert np.isfinite(
    list(test_metrics.values())
).all()

print("\nEvaluation functions: READY")

VERSION 2 — EVALUATION FUNCTIONS
poisson_deviance         : 0.592355
mae                      : 0.475000
rmse                     : 0.512348
mean_bias                : 0.025000
actual_mean_goals        : 0.750000
predicted_mean_goals     : 0.775000

Evaluation functions: READY


In [8]:
# Reproduce the selected Version 1 XGBoost model

import time
import xgboost as xgb

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline


def build_xgboost_preprocessor(feature_set):
    """
    Construct the original XGBoost preprocessing pipeline.
    Numerical features pass through unchanged.
    Categorical features are one-hot encoded.
    """

    numeric = [
        feature for feature in feature_set
        if feature in NUMERIC_FEATURES
    ]

    categorical = [
        feature for feature in feature_set
        if feature in CATEGORICAL_FEATURES
    ]

    return ColumnTransformer(
        transformers=[
            ("numeric", "passthrough", numeric),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                ),
                categorical,
            ),
        ],
        remainder="drop",
    )


XGB_FEATURES = FEATURE_SETS["context"]

XGB_V1_PARAMS = {
    "objective": "count:poisson",
    "n_estimators": 15,
    "max_depth": 2,
    "learning_rate": 0.03,
    "min_child_weight": 10,
    "reg_lambda": 20,
    "random_state": 42,
    "n_jobs": -1,
}

xgb_oof_parts = []

start_time = time.perf_counter()

for fold in temporal_folds:
    train = fold["train"]
    val = fold["val"]

    model = Pipeline(
        steps=[
            (
                "preprocessor",
                build_xgboost_preprocessor(XGB_FEATURES),
            ),
            (
                "model",
                xgb.XGBRegressor(**XGB_V1_PARAMS),
            ),
        ]
    )

    model.fit(
        train[XGB_FEATURES],
        train[TARGET],
    )

    predictions = np.clip(
        model.predict(val[XGB_FEATURES]),
        1e-9,
        None,
    )

    xgb_oof_parts.append(
        pd.DataFrame({
            "match_id": val["match_id"].values,
            "date": val["date"].values,
            "fold": fold["name"],
            "y_true": val[TARGET].values,
            "y_pred": predictions,
        })
    )

xgb_v1_oof = pd.concat(
    xgb_oof_parts,
    ignore_index=True,
)

xgb_v1_results = summarize_oof_predictions(
    xgb_v1_oof
)

elapsed = time.perf_counter() - start_time

print("VERSION 1 XGBOOST REPRODUCTION")
print("=" * 65)

print(
    xgb_v1_results["fold_results"][
        ["fold", "matches", "poisson_deviance", "mae", "mean_bias"]
    ].round(6).to_string(index=False)
)

macro_deviance = xgb_v1_results[
    "macro_metrics"
]["poisson_deviance"]

print(f"\nMacro Poisson deviance: {macro_deviance:.6f}")
print("Original reference:     1.109154")
print(f"Difference:             {macro_deviance - 1.109154:+.6f}")
print(f"Training time:          {elapsed:.2f} seconds")

VERSION 1 XGBOOST REPRODUCTION
  fold  matches  poisson_deviance      mae  mean_bias
Fold 1      158          1.047402 0.691808  -0.041778
Fold 2       72          1.241586 0.725026   0.230649
Fold 3       59          1.136058 0.755488  -0.202619
Fold 4       51          1.006316 0.674880  -0.070036

Macro Poisson deviance: 1.107840
Original reference:     1.109154
Difference:             -0.001314
Training time:          0.70 seconds


In [10]:
# Compare reproduced XGBoost predictions with saved Version 1 predictions

saved_oof = pd.read_csv(
    ARTIFACT_DIR / "main_models_oof.csv"
)

comparison = xgb_v1_oof.merge(
    saved_oof[["match_id", "xgb_pred"]],
    on="match_id",
    how="inner",
    validate="one_to_one",
)

comparison["absolute_difference"] = np.abs(
    comparison["y_pred"] - comparison["xgb_pred"]
)

print("XGBOOST REPRODUCTION — PREDICTION COMPARISON")
print("=" * 65)

print("Matched appearances:", len(comparison))

print(
    "Mean absolute prediction difference:",
    round(comparison["absolute_difference"].mean(), 8)
)

print(
    "Maximum absolute prediction difference:",
    round(comparison["absolute_difference"].max(), 8)
)

print(
    "\nOriginal prediction range:",
    (
        round(comparison["xgb_pred"].min(), 6),
        round(comparison["xgb_pred"].max(), 6),
    )
)

print(
    "Reproduced prediction range:",
    (
        round(comparison["y_pred"].min(), 6),
        round(comparison["y_pred"].max(), 6),
    )
)

print("\nFOLD-LEVEL DIFFERENCES")

print(
    comparison.groupby("fold").agg(
        matches=("match_id", "size"),
        mean_absolute_difference=("absolute_difference", "mean"),
        max_absolute_difference=("absolute_difference", "max"),
    ).round(8).to_string()
)

assert len(comparison) == 340

XGBOOST REPRODUCTION — PREDICTION COMPARISON
Matched appearances: 340
Mean absolute prediction difference: 0.01160396
Maximum absolute prediction difference: 0.03623834

Original prediction range: (0.694734, 0.797634)
Reproduced prediction range: (0.695189, 0.796611)

FOLD-LEVEL DIFFERENCES
        matches  mean_absolute_difference  max_absolute_difference
fold                                                              
Fold 1      158                  0.013824                 0.036238
Fold 2       72                  0.013501                 0.020320
Fold 3       59                  0.005277                 0.033913
Fold 4       51                  0.009369                 0.019647


In [12]:
# Inspect the frozen Version 1 XGBoost model specification

spec_path = ARTIFACT_DIR / "main_model_spec.json"

print("VERSION 1 — MODEL SPECIFICATION")
print("=" * 65)

if spec_path.exists():

    with open(spec_path, encoding="utf-8") as f:
        original_spec = json.load(f)

    print(
        json.dumps(
            original_spec,
            indent=2,
            default=str,
        )
    )

else:
    print("main_model_spec.json was not found.")

    print("\nAvailable model-related artifacts:")

    for path in sorted(ARTIFACT_DIR.glob("*model*")):
        print("-", path.name)

print("\nCurrent reproduction parameters:")
print(json.dumps(XGB_V1_PARAMS, indent=2))

VERSION 1 — MODEL SPECIFICATION
{
  "phase": "Phase 7 - Main Models",
  "selection_policy": {
    "primary_metric": "macro_poisson_deviance",
    "secondary_checks": [
      "pooled_poisson_deviance",
      "mae",
      "rmse",
      "mean_bias",
      "fold_stability"
    ],
    "final_test_used_for_selection": false
  },
  "selected_ensemble": {
    "xgboost_weight": 0.6,
    "catboost_weight": 0.4,
    "macro_poisson_deviance": 1.106496,
    "pooled_poisson_deviance": 1.096307,
    "pooled_mae": 0.701696,
    "pooled_bias": 0.007642
  },
  "xgboost": {
    "feature_set": "context",
    "excluded_identity_categoricals": [
      "season",
      "team",
      "opponent",
      "competition"
    ],
    "params": {
      "objective": "count:poisson",
      "n_estimators": 15,
      "max_depth": 2,
      "learning_rate": 0.03,
      "min_child_weight": 10.0,
      "subsample": 0.9,
      "colsample_bytree": 0.8,
      "reg_alpha": 0.0,
      "reg_lambda": 20.0,
      "random_state": 42,
 

In [14]:
# Reproduce XGBoost using the exact frozen Version 1 parameters

XGB_V1_PARAMS = original_spec["xgboost"]["params"].copy()

xgb_oof_parts = []

start_time = time.perf_counter()

for fold in temporal_folds:

    train = fold["train"]
    val = fold["val"]

    model = Pipeline(
        steps=[
            (
                "preprocessor",
                build_xgboost_preprocessor(XGB_FEATURES),
            ),
            (
                "model",
                xgb.XGBRegressor(**XGB_V1_PARAMS),
            ),
        ]
    )

    model.fit(
        train[XGB_FEATURES],
        train[TARGET],
    )

    predictions = np.clip(
        model.predict(val[XGB_FEATURES]),
        1e-9,
        None,
    )

    xgb_oof_parts.append(
        pd.DataFrame({
            "match_id": val["match_id"].values,
            "date": val["date"].values,
            "fold": fold["name"],
            "y_true": val[TARGET].values,
            "y_pred": predictions,
        })
    )

xgb_v1_oof = pd.concat(
    xgb_oof_parts,
    ignore_index=True,
)

xgb_v1_results = summarize_oof_predictions(
    xgb_v1_oof
)

# Compare against the original saved predictions
comparison = xgb_v1_oof.merge(
    saved_oof[["match_id", "xgb_pred"]],
    on="match_id",
    validate="one_to_one",
)

absolute_difference = np.abs(
    comparison["y_pred"] - comparison["xgb_pred"]
)

macro_deviance = xgb_v1_results[
    "macro_metrics"
]["poisson_deviance"]

original_deviance = original_spec[
    "xgboost"
]["macro_poisson_deviance"]

print("CORRECTED XGBOOST REPRODUCTION")
print("=" * 65)

print(f"Original macro deviance:   {original_deviance:.6f}")
print(f"Reproduced macro deviance: {macro_deviance:.6f}")
print(f"Metric difference:         {macro_deviance - original_deviance:+.8f}")

print(
    f"\nMean prediction difference: "
    f"{absolute_difference.mean():.8f}"
)

print(
    f"Maximum difference:         "
    f"{absolute_difference.max():.8f}"
)

print(
    f"Training time:              "
    f"{time.perf_counter() - start_time:.2f} seconds"
)

assert len(comparison) == 340

CORRECTED XGBOOST REPRODUCTION
Original macro deviance:   1.109154
Reproduced macro deviance: 1.109154
Metric difference:         -0.00000010

Mean prediction difference: 0.00000001
Maximum difference:         0.00000003
Training time:              0.46 seconds


In [16]:
# Reproduce the selected Version 1 CatBoost model

from catboost import CatBoostRegressor

CAT_FEATURES = FEATURE_SETS["full"]

CAT_V1_PARAMS = original_spec["catboost"]["params"].copy()

CAT_CATEGORICAL = [
    feature for feature in CAT_FEATURES
    if feature in CATEGORICAL_FEATURES
]

cat_oof_parts = []

start_time = time.perf_counter()

for fold in temporal_folds:

    train = fold["train"]
    val = fold["val"]

    X_train = train[CAT_FEATURES].copy()
    X_val = val[CAT_FEATURES].copy()

    # Apply the original categorical preprocessing
    for feature in CAT_CATEGORICAL:
        X_train[feature] = (
            X_train[feature]
            .astype(str)
            .fillna("__MISSING__")
        )

        X_val[feature] = (
            X_val[feature]
            .astype(str)
            .fillna("__MISSING__")
        )

    model = CatBoostRegressor(**CAT_V1_PARAMS)

    model.fit(
        X_train,
        train[TARGET],
        cat_features=CAT_CATEGORICAL,
    )

    predictions = np.clip(
        model.predict(X_val),
        1e-9,
        None,
    )

    cat_oof_parts.append(
        pd.DataFrame({
            "match_id": val["match_id"].values,
            "date": val["date"].values,
            "fold": fold["name"],
            "y_true": val[TARGET].values,
            "y_pred": predictions,
        })
    )

cat_v1_oof = pd.concat(
    cat_oof_parts,
    ignore_index=True,
)

cat_v1_results = summarize_oof_predictions(
    cat_v1_oof
)

# Compare against frozen Version 1 predictions
cat_comparison = cat_v1_oof.merge(
    saved_oof[["match_id", "catboost_pred"]],
    on="match_id",
    validate="one_to_one",
)

cat_difference = np.abs(
    cat_comparison["y_pred"]
    - cat_comparison["catboost_pred"]
)

macro_deviance = cat_v1_results[
    "macro_metrics"
]["poisson_deviance"]

original_deviance = original_spec[
    "catboost"
]["macro_poisson_deviance"]

print("CATBOOST VERSION 1 REPRODUCTION")
print("=" * 65)

print(f"Original macro deviance:   {original_deviance:.6f}")
print(f"Reproduced macro deviance: {macro_deviance:.6f}")
print(f"Metric difference:         {macro_deviance - original_deviance:+.8f}")

print(
    f"\nMean prediction difference: "
    f"{cat_difference.mean():.8f}"
)

print(
    f"Maximum difference:         "
    f"{cat_difference.max():.8f}"
)

print(
    f"Training time:              "
    f"{time.perf_counter() - start_time:.2f} seconds"
)

assert len(cat_comparison) == 340

CATBOOST VERSION 1 REPRODUCTION
Original macro deviance:   1.112413
Reproduced macro deviance: 1.112413
Metric difference:         +0.00000038

Mean prediction difference: 0.00000000
Maximum difference:         0.00000000
Training time:              10.85 seconds


In [18]:
# Reconstruct and verify the original Version 1 ensemble

ensemble_weights = original_spec["selected_ensemble"]

xgb_weight = ensemble_weights["xgboost_weight"]
cat_weight = ensemble_weights["catboost_weight"]

assert np.isclose(xgb_weight + cat_weight, 1.0)

# Align both models by match ID
ensemble_oof = xgb_v1_oof.merge(
    cat_v1_oof[["match_id", "y_pred"]],
    on="match_id",
    how="inner",
    suffixes=("_xgb", "_cat"),
    validate="one_to_one",
)

# Reconstruct the original weighted ensemble
ensemble_oof["y_pred"] = (
    xgb_weight * ensemble_oof["y_pred_xgb"]
    + cat_weight * ensemble_oof["y_pred_cat"]
)

ensemble_v1_results = summarize_oof_predictions(
    ensemble_oof
)

# Compare against the original saved ensemble predictions
ensemble_comparison = ensemble_oof.merge(
    saved_oof[["match_id", "blend_pred"]],
    on="match_id",
    validate="one_to_one",
)

ensemble_difference = np.abs(
    ensemble_comparison["y_pred"]
    - ensemble_comparison["blend_pred"]
)

original_macro = ensemble_weights["macro_poisson_deviance"]

reproduced_macro = ensemble_v1_results[
    "macro_metrics"
]["poisson_deviance"]

print("VERSION 1 — ENSEMBLE REPRODUCTION")
print("=" * 65)

print(f"XGBoost weight:           {xgb_weight:.0%}")
print(f"CatBoost weight:          {cat_weight:.0%}")

print(f"\nOriginal macro deviance:   {original_macro:.6f}")
print(f"Reproduced macro deviance: {reproduced_macro:.6f}")
print(f"Metric difference:         {reproduced_macro - original_macro:+.8f}")

print(
    f"\nMean prediction difference: "
    f"{ensemble_difference.mean():.8f}"
)

print(
    f"Maximum difference:         "
    f"{ensemble_difference.max():.8f}"
)

print("\nFOLD-LEVEL PERFORMANCE")

print(
    ensemble_v1_results["fold_results"][
        ["fold", "matches", "poisson_deviance", "mae", "mean_bias"]
    ].round(6).to_string(index=False)
)

assert len(ensemble_oof) == 340
assert np.isclose(reproduced_macro, original_macro, atol=1e-5)

VERSION 1 — ENSEMBLE REPRODUCTION
XGBoost weight:           60%
CatBoost weight:          40%

Original macro deviance:   1.106496
Reproduced macro deviance: 1.106496
Metric difference:         +0.00000045

Mean prediction difference: 0.00000000
Maximum difference:         0.00000000

FOLD-LEVEL PERFORMANCE
  fold  matches  poisson_deviance      mae  mean_bias
Fold 1      158          1.044600 0.682861   0.002623
Fold 2       72          1.238840 0.725466   0.238552
Fold 3       59          1.153896 0.756635  -0.224619
Fold 4       51          0.988650 0.662932  -0.034104


In [20]:
# Experiment 1: Evaluate XGBoost tree depth

depth_candidates = [2, 3, 4]

depth_experiment_results = []
depth_oof_predictions = {}

start_time = time.perf_counter()

for depth in depth_candidates:

    params = XGB_V1_PARAMS.copy()
    params["max_depth"] = depth

    fold_predictions = []

    for fold in temporal_folds:

        train = fold["train"]
        val = fold["val"]

        model = Pipeline(
            steps=[
                (
                    "preprocessor",
                    build_xgboost_preprocessor(XGB_FEATURES),
                ),
                (
                    "model",
                    xgb.XGBRegressor(**params),
                ),
            ]
        )

        model.fit(
            train[XGB_FEATURES],
            train[TARGET],
        )

        predictions = np.clip(
            model.predict(val[XGB_FEATURES]),
            1e-9,
            None,
        )

        fold_predictions.append(
            pd.DataFrame({
                "match_id": val["match_id"].values,
                "fold": fold["name"],
                "y_true": val[TARGET].values,
                "y_pred": predictions,
            })
        )

    candidate_oof = pd.concat(
        fold_predictions,
        ignore_index=True,
    )

    depth_oof_predictions[depth] = candidate_oof

    summary = summarize_oof_predictions(
        candidate_oof
    )

    depth_experiment_results.append({
        "max_depth": depth,
        "macro_poisson_deviance": summary[
            "macro_metrics"
        ]["poisson_deviance"],
        "pooled_poisson_deviance": summary[
            "pooled_metrics"
        ]["poisson_deviance"],
        "macro_mae": summary["macro_metrics"]["mae"],
        "macro_rmse": summary["macro_metrics"]["rmse"],
        "macro_bias": summary["macro_metrics"]["mean_bias"],
    })

depth_results_df = pd.DataFrame(
    depth_experiment_results
).sort_values("macro_poisson_deviance")

print("XGBOOST EXPERIMENT 1 — TREE DEPTH")
print("=" * 85)

print(
    depth_results_df.round(6).to_string(index=False)
)

print(
    f"\nExperiment time: "
    f"{time.perf_counter() - start_time:.2f} seconds"
)

# Confirm the depth-2 candidate reproduces the verified baseline
baseline_depth = depth_results_df.loc[
    depth_results_df["max_depth"] == 2,
    "macro_poisson_deviance",
].iloc[0]

assert np.isclose(
    baseline_depth,
    xgb_v1_results["macro_metrics"]["poisson_deviance"],
    atol=1e-6,
)

XGBOOST EXPERIMENT 1 — TREE DEPTH
 max_depth  macro_poisson_deviance  pooled_poisson_deviance  macro_mae  macro_rmse  macro_bias
         2                1.109154                 1.098756   0.712476    0.869278   -0.015777
         3                1.110045                 1.099673   0.711300    0.869660   -0.008993
         4                1.113508                 1.102080   0.713058    0.871132   -0.011300

Experiment time: 1.41 seconds


In [22]:
# Compare XGBoost tree depths across chronological folds

fold_comparison = []

for depth, oof in depth_oof_predictions.items():

    summary = summarize_oof_predictions(oof)

    for _, row in summary["fold_results"].iterrows():

        fold_comparison.append({
            "max_depth": depth,
            "fold": row["fold"],
            "poisson_deviance": row["poisson_deviance"],
            "mae": row["mae"],
            "mean_bias": row["mean_bias"],
        })

fold_comparison_df = pd.DataFrame(fold_comparison)

# Compare each candidate against the original depth-2 model
deviance_pivot = fold_comparison_df.pivot(
    index="fold",
    columns="max_depth",
    values="poisson_deviance",
)

deviance_changes = deviance_pivot.subtract(
    deviance_pivot[2],
    axis=0,
)

print("XGBOOST TREE DEPTH — FOLD STABILITY")
print("=" * 70)

print("\nPoisson deviance by fold:")
print(deviance_pivot.round(6).to_string())

print("\nChange relative to depth 2:")
print(deviance_changes.round(6).to_string())

print("\nNumber of folds improved:")

for depth in [3, 4]:

    improvements = (deviance_changes[depth] < 0).sum()

    print(
        f"Depth {depth}: {improvements} of 4 folds"
    )

XGBOOST TREE DEPTH — FOLD STABILITY

Poisson deviance by fold:
max_depth         2         3         4
fold                                   
Fold 1     1.047050  1.047007  1.045635
Fold 2     1.246801  1.253219  1.263297
Fold 3     1.136577  1.136516  1.142437
Fold 4     1.006188  1.003440  1.002662

Change relative to depth 2:
max_depth    2         3         4
fold                              
Fold 1     0.0 -0.000043 -0.001415
Fold 2     0.0  0.006418  0.016497
Fold 3     0.0 -0.000061  0.005860
Fold 4     0.0 -0.002748 -0.003526

Number of folds improved:
Depth 3: 3 of 4 folds
Depth 4: 2 of 4 folds


In [24]:
# Experiment 2: Test XGBoost regularization strength

regularization_candidates = {
    "original": {
        "min_child_weight": 10,
        "reg_lambda": 20,
    },
    "child_weight_5": {
        "min_child_weight": 5,
        "reg_lambda": 20,
    },
    "child_weight_2": {
        "min_child_weight": 2,
        "reg_lambda": 20,
    },
    "lambda_10": {
        "min_child_weight": 10,
        "reg_lambda": 10,
    },
    "lambda_5": {
        "min_child_weight": 10,
        "reg_lambda": 5,
    },
}

regularization_results = []
regularization_oof = {}

start_time = time.perf_counter()

for candidate_name, changes in regularization_candidates.items():

    params = XGB_V1_PARAMS.copy()
    params.update(changes)

    fold_predictions = []

    for fold in temporal_folds:

        train = fold["train"]
        val = fold["val"]

        model = Pipeline(
            steps=[
                (
                    "preprocessor",
                    build_xgboost_preprocessor(XGB_FEATURES),
                ),
                (
                    "model",
                    xgb.XGBRegressor(**params),
                ),
            ]
        )

        model.fit(
            train[XGB_FEATURES],
            train[TARGET],
        )

        predictions = np.clip(
            model.predict(val[XGB_FEATURES]),
            1e-9,
            None,
        )

        fold_predictions.append(
            pd.DataFrame({
                "match_id": val["match_id"].values,
                "fold": fold["name"],
                "y_true": val[TARGET].values,
                "y_pred": predictions,
            })
        )

    candidate_oof = pd.concat(
        fold_predictions,
        ignore_index=True,
    )

    regularization_oof[candidate_name] = candidate_oof

    summary = summarize_oof_predictions(candidate_oof)

    regularization_results.append({
        "candidate": candidate_name,
        "min_child_weight": changes["min_child_weight"],
        "reg_lambda": changes["reg_lambda"],
        "macro_deviance": summary[
            "macro_metrics"
        ]["poisson_deviance"],
        "pooled_deviance": summary[
            "pooled_metrics"
        ]["poisson_deviance"],
        "macro_mae": summary["macro_metrics"]["mae"],
        "macro_bias": summary["macro_metrics"]["mean_bias"],
    })

regularization_results_df = pd.DataFrame(
    regularization_results
).sort_values("macro_deviance")

print("XGBOOST EXPERIMENT 2 — REGULARIZATION")
print("=" * 95)

print(
    regularization_results_df.round(6).to_string(index=False)
)

print(
    f"\nExperiment time: "
    f"{time.perf_counter() - start_time:.2f} seconds"
)

# Confirm the original candidate matches the verified baseline
original_result = regularization_results_df.loc[
    regularization_results_df["candidate"] == "original",
    "macro_deviance",
].iloc[0]

assert np.isclose(
    original_result,
    xgb_v1_results["macro_metrics"]["poisson_deviance"],
    atol=1e-6,
)

XGBOOST EXPERIMENT 2 — REGULARIZATION
     candidate  min_child_weight  reg_lambda  macro_deviance  pooled_deviance  macro_mae  macro_bias
     lambda_10                10          10        1.109054         1.098606   0.712389   -0.015577
      original                10          20        1.109154         1.098756   0.712476   -0.015777
      lambda_5                10           5        1.109157         1.098647   0.712534   -0.015968
child_weight_5                 5          20        1.109196         1.098782   0.712483   -0.015776
child_weight_2                 2          20        1.109196         1.098782   0.712483   -0.015776

Experiment time: 2.10 seconds


In [26]:
# Experiment 3: Test XGBoost boosting iterations

iteration_candidates = [15, 30, 50, 100, 200]

iteration_results = []
iteration_oof = {}

start_time = time.perf_counter()

for n_estimators in iteration_candidates:

    params = XGB_V1_PARAMS.copy()
    params["n_estimators"] = n_estimators

    fold_predictions = []

    for fold in temporal_folds:

        train = fold["train"]
        val = fold["val"]

        model = Pipeline(
            steps=[
                (
                    "preprocessor",
                    build_xgboost_preprocessor(XGB_FEATURES),
                ),
                (
                    "model",
                    xgb.XGBRegressor(**params),
                ),
            ]
        )

        model.fit(
            train[XGB_FEATURES],
            train[TARGET],
        )

        predictions = np.clip(
            model.predict(val[XGB_FEATURES]),
            1e-9,
            None,
        )

        fold_predictions.append(
            pd.DataFrame({
                "match_id": val["match_id"].values,
                "fold": fold["name"],
                "y_true": val[TARGET].values,
                "y_pred": predictions,
            })
        )

    candidate_oof = pd.concat(
        fold_predictions,
        ignore_index=True,
    )

    iteration_oof[n_estimators] = candidate_oof

    summary = summarize_oof_predictions(
        candidate_oof
    )

    iteration_results.append({
        "n_estimators": n_estimators,
        "macro_deviance": summary[
            "macro_metrics"
        ]["poisson_deviance"],
        "pooled_deviance": summary[
            "pooled_metrics"
        ]["poisson_deviance"],
        "macro_mae": summary[
            "macro_metrics"
        ]["mae"],
        "macro_rmse": summary[
            "macro_metrics"
        ]["rmse"],
        "macro_bias": summary[
            "macro_metrics"
        ]["mean_bias"],
        "prediction_std": candidate_oof["y_pred"].std(),
    })

iteration_results_df = pd.DataFrame(
    iteration_results
).sort_values("macro_deviance")

print("XGBOOST EXPERIMENT 3 — BOOSTING ITERATIONS")
print("=" * 105)

print(
    iteration_results_df.round(6).to_string(index=False)
)

print(
    f"\nExperiment time: "
    f"{time.perf_counter() - start_time:.2f} seconds"
)

# Verify that 15 iterations reproduce the original baseline
baseline_iterations = iteration_results_df.loc[
    iteration_results_df["n_estimators"] == 15,
    "macro_deviance",
].iloc[0]

assert np.isclose(
    baseline_iterations,
    xgb_v1_results["macro_metrics"]["poisson_deviance"],
    atol=1e-6,
)

XGBOOST EXPERIMENT 3 — BOOSTING ITERATIONS
 n_estimators  macro_deviance  pooled_deviance  macro_mae  macro_rmse  macro_bias  prediction_std
           30        1.107376         1.096395   0.708606    0.868314    0.002390        0.056586
           15        1.109154         1.098756   0.712476    0.869278   -0.015777        0.029689
           50        1.109832         1.098545   0.705113    0.869362    0.025949        0.082072
          100        1.120768         1.110028   0.705923    0.874222    0.056696        0.133109
          200        1.137584         1.132856   0.718085    0.882754    0.085374        0.203060

Experiment time: 5.87 seconds


In [28]:
# Evaluate the 30-tree XGBoost candidate and its ensemble impact

xgb_30_oof = iteration_oof[30].copy()

# Combine the updated XGBoost predictions with original CatBoost
updated_ensemble_oof = xgb_30_oof.merge(
    cat_v1_oof[["match_id", "y_pred"]],
    on="match_id",
    how="inner",
    suffixes=("_xgb", "_cat"),
    validate="one_to_one",
)

updated_ensemble_oof["y_pred"] = (
    0.6 * updated_ensemble_oof["y_pred_xgb"]
    + 0.4 * updated_ensemble_oof["y_pred_cat"]
)

# Evaluate both the updated XGBoost and ensemble
xgb_30_results = summarize_oof_predictions(xgb_30_oof)

updated_ensemble_results = summarize_oof_predictions(
    updated_ensemble_oof
)

# Compare against the verified Version 1 baseline
comparison_rows = []

for name, results in [
    ("Original XGBoost", xgb_v1_results),
    ("XGBoost 30 trees", xgb_30_results),
    ("Original ensemble", ensemble_v1_results),
    ("Updated ensemble", updated_ensemble_results),
]:
    comparison_rows.append({
        "model": name,
        "macro_deviance": results["macro_metrics"]["poisson_deviance"],
        "pooled_deviance": results["pooled_metrics"]["poisson_deviance"],
        "macro_mae": results["macro_metrics"]["mae"],
        "macro_bias": results["macro_metrics"]["mean_bias"],
    })

ensemble_comparison_df = pd.DataFrame(comparison_rows)

print("XGBOOST 30 TREES — ENSEMBLE EVALUATION")
print("=" * 85)

print(
    ensemble_comparison_df.round(6).to_string(index=False)
)

print("\nFOLD-LEVEL ENSEMBLE COMPARISON")

original_folds = ensemble_v1_results["fold_results"].set_index("fold")
updated_folds = updated_ensemble_results["fold_results"].set_index("fold")

fold_changes = pd.DataFrame({
    "original_deviance": original_folds["poisson_deviance"],
    "updated_deviance": updated_folds["poisson_deviance"],
})

fold_changes["change"] = (
    fold_changes["updated_deviance"]
    - fold_changes["original_deviance"]
)

print(fold_changes.round(6).to_string())

print(
    "\nFolds improved:",
    int((fold_changes["change"] < 0).sum()),
    "of 4"
)

print(
    "Macro ensemble improvement:",
    round(
        ensemble_v1_results["macro_metrics"]["poisson_deviance"]
        - updated_ensemble_results["macro_metrics"]["poisson_deviance"],
        6,
    )
)

assert len(updated_ensemble_oof) == 340

XGBOOST 30 TREES — ENSEMBLE EVALUATION
            model  macro_deviance  pooled_deviance  macro_mae  macro_bias
 Original XGBoost        1.109154         1.098756   0.712476   -0.015777
 XGBoost 30 trees        1.107376         1.096395   0.708606    0.002390
Original ensemble        1.106496         1.096307   0.706974   -0.004387
 Updated ensemble        1.106997         1.096566   0.704651    0.006513

FOLD-LEVEL ENSEMBLE COMPARISON
        original_deviance  updated_deviance    change
fold                                                 
Fold 1           1.044600          1.044042 -0.000558
Fold 2           1.238840          1.240409  0.001569
Fold 3           1.153896          1.155507  0.001611
Fold 4           0.988650          0.988031 -0.000619

Folds improved: 2 of 4
Macro ensemble improvement: -0.000501


In [30]:
# Experiment 4: Evaluate CatBoost tree depth

cat_depth_candidates = [2, 3, 4]

cat_depth_results = []
cat_depth_oof = {}

start_time = time.perf_counter()

for depth in cat_depth_candidates:

    params = CAT_V1_PARAMS.copy()
    params["depth"] = depth

    fold_predictions = []

    for fold in temporal_folds:

        train = fold["train"]
        val = fold["val"]

        X_train = train[CAT_FEATURES].copy()
        X_val = val[CAT_FEATURES].copy()

        # Preserve the original categorical preprocessing
        for feature in CAT_CATEGORICAL:
            X_train[feature] = (
                X_train[feature]
                .astype(str)
                .fillna("__MISSING__")
            )

            X_val[feature] = (
                X_val[feature]
                .astype(str)
                .fillna("__MISSING__")
            )

        model = CatBoostRegressor(**params)

        model.fit(
            X_train,
            train[TARGET],
            cat_features=CAT_CATEGORICAL,
        )

        predictions = np.clip(
            model.predict(X_val),
            1e-9,
            None,
        )

        fold_predictions.append(
            pd.DataFrame({
                "match_id": val["match_id"].values,
                "fold": fold["name"],
                "y_true": val[TARGET].values,
                "y_pred": predictions,
            })
        )

    candidate_oof = pd.concat(
        fold_predictions,
        ignore_index=True,
    )

    cat_depth_oof[depth] = candidate_oof

    summary = summarize_oof_predictions(candidate_oof)

    cat_depth_results.append({
        "depth": depth,
        "macro_deviance": summary[
            "macro_metrics"
        ]["poisson_deviance"],
        "pooled_deviance": summary[
            "pooled_metrics"
        ]["poisson_deviance"],
        "macro_mae": summary[
            "macro_metrics"
        ]["mae"],
        "macro_rmse": summary[
            "macro_metrics"
        ]["rmse"],
        "macro_bias": summary[
            "macro_metrics"
        ]["mean_bias"],
        "prediction_std": candidate_oof["y_pred"].std(),
    })

cat_depth_results_df = pd.DataFrame(
    cat_depth_results
).sort_values("macro_deviance")

print("CATBOOST EXPERIMENT 4 — TREE DEPTH")
print("=" * 100)

print(
    cat_depth_results_df.round(6).to_string(index=False)
)

print(
    f"\nExperiment time: "
    f"{time.perf_counter() - start_time:.2f} seconds"
)

# Confirm depth 2 reproduces the verified CatBoost baseline
baseline_depth = cat_depth_results_df.loc[
    cat_depth_results_df["depth"] == 2,
    "macro_deviance",
].iloc[0]

assert np.isclose(
    baseline_depth,
    cat_v1_results["macro_metrics"]["poisson_deviance"],
    atol=1e-6,
)

CATBOOST EXPERIMENT 4 — TREE DEPTH
 depth  macro_deviance  pooled_deviance  macro_mae  macro_rmse  macro_bias  prediction_std
     2        1.112413         1.102292   0.698919    0.868244    0.012699        0.123679
     3        1.124673         1.110185   0.703087    0.871734   -0.009295        0.146843
     4        1.126931         1.113147   0.702643    0.873570    0.034603        0.144087

Experiment time: 45.42 seconds


In [32]:
# Experiment 5: Evaluate CatBoost boosting iterations

cat_iteration_candidates = [50, 100, 150, 200]

cat_iteration_results = []
cat_iteration_oof = {}

start_time = time.perf_counter()

for iterations in cat_iteration_candidates:

    params = CAT_V1_PARAMS.copy()
    params["iterations"] = iterations

    fold_predictions = []

    for fold in temporal_folds:

        train = fold["train"]
        val = fold["val"]

        X_train = train[CAT_FEATURES].copy()
        X_val = val[CAT_FEATURES].copy()

        # Preserve the original categorical preprocessing
        for feature in CAT_CATEGORICAL:
            X_train[feature] = (
                X_train[feature]
                .astype(str)
                .fillna("__MISSING__")
            )

            X_val[feature] = (
                X_val[feature]
                .astype(str)
                .fillna("__MISSING__")
            )

        model = CatBoostRegressor(**params)

        model.fit(
            X_train,
            train[TARGET],
            cat_features=CAT_CATEGORICAL,
        )

        predictions = np.clip(
            model.predict(X_val),
            1e-9,
            None,
        )

        fold_predictions.append(
            pd.DataFrame({
                "match_id": val["match_id"].values,
                "fold": fold["name"],
                "y_true": val[TARGET].values,
                "y_pred": predictions,
            })
        )

    candidate_oof = pd.concat(
        fold_predictions,
        ignore_index=True,
    )

    cat_iteration_oof[iterations] = candidate_oof

    summary = summarize_oof_predictions(candidate_oof)

    cat_iteration_results.append({
        "iterations": iterations,
        "macro_deviance": summary[
            "macro_metrics"
        ]["poisson_deviance"],
        "pooled_deviance": summary[
            "pooled_metrics"
        ]["poisson_deviance"],
        "macro_mae": summary[
            "macro_metrics"
        ]["mae"],
        "macro_rmse": summary[
            "macro_metrics"
        ]["rmse"],
        "macro_bias": summary[
            "macro_metrics"
        ]["mean_bias"],
        "prediction_std": candidate_oof["y_pred"].std(),
    })

cat_iteration_results_df = pd.DataFrame(
    cat_iteration_results
).sort_values("macro_deviance")

print("CATBOOST EXPERIMENT 5 — BOOSTING ITERATIONS")
print("=" * 105)

print(
    cat_iteration_results_df.round(6).to_string(index=False)
)

print(
    f"\nExperiment time: "
    f"{time.perf_counter() - start_time:.2f} seconds"
)

# Confirm 100 iterations reproduce the original CatBoost baseline
baseline_iterations = cat_iteration_results_df.loc[
    cat_iteration_results_df["iterations"] == 100,
    "macro_deviance",
].iloc[0]

assert np.isclose(
    baseline_iterations,
    cat_v1_results["macro_metrics"]["poisson_deviance"],
    atol=1e-6,
)

CATBOOST EXPERIMENT 5 — BOOSTING ITERATIONS
 iterations  macro_deviance  pooled_deviance  macro_mae  macro_rmse  macro_bias  prediction_std
        150        1.107921         1.098969   0.696770    0.865840    0.005427        0.141846
         50        1.112247         1.105657   0.699296    0.870297    0.055050        0.099894
        100        1.112413         1.102292   0.698919    0.868244    0.012699        0.123679
        200        1.129687         1.113024   0.703800    0.872176   -0.015866        0.134055

Experiment time: 52.29 seconds


In [34]:
# Experiment 5: Evaluate updated CatBoost ensemble combinations

cat_150_oof = cat_iteration_oof[150].copy()

# Align predictions using match IDs
ensemble_candidates = (
    xgb_v1_oof[
        ["match_id", "fold", "y_true", "y_pred"]
    ]
    .rename(columns={"y_pred": "xgb_original"})
    .merge(
        iteration_oof[30][["match_id", "y_pred"]]
        .rename(columns={"y_pred": "xgb_30"}),
        on="match_id",
        validate="one_to_one",
    )
    .merge(
        cat_v1_oof[["match_id", "y_pred"]]
        .rename(columns={"y_pred": "cat_original"}),
        on="match_id",
        validate="one_to_one",
    )
    .merge(
        cat_150_oof[["match_id", "y_pred"]]
        .rename(columns={"y_pred": "cat_150"}),
        on="match_id",
        validate="one_to_one",
    )
)

assert len(ensemble_candidates) == 340

# Preserve the original ensemble weights
ensemble_configurations = {
    "Original ensemble": ("xgb_original", "cat_original"),
    "Original XGB + Cat150": ("xgb_original", "cat_150"),
    "XGB30 + Cat150": ("xgb_30", "cat_150"),
}

ensemble_candidate_results = {}
ensemble_candidate_rows = []

for name, (xgb_col, cat_col) in ensemble_configurations.items():

    candidate_oof = ensemble_candidates[
        ["match_id", "fold", "y_true"]
    ].copy()

    candidate_oof["y_pred"] = (
        0.6 * ensemble_candidates[xgb_col]
        + 0.4 * ensemble_candidates[cat_col]
    )

    summary = summarize_oof_predictions(candidate_oof)

    ensemble_candidate_results[name] = summary

    ensemble_candidate_rows.append({
        "configuration": name,
        "macro_deviance": summary[
            "macro_metrics"
        ]["poisson_deviance"],
        "pooled_deviance": summary[
            "pooled_metrics"
        ]["poisson_deviance"],
        "macro_mae": summary[
            "macro_metrics"
        ]["mae"],
        "macro_bias": summary[
            "macro_metrics"
        ]["mean_bias"],
    })

ensemble_candidate_df = pd.DataFrame(
    ensemble_candidate_rows
).sort_values("macro_deviance")

print("CATBOOST 150 — ENSEMBLE COMPARISON")
print("=" * 100)

print(
    ensemble_candidate_df.round(6).to_string(index=False)
)

# Compare each candidate against the original ensemble by fold
original_folds = ensemble_candidate_results[
    "Original ensemble"
]["fold_results"].set_index("fold")

for name in [
    "Original XGB + Cat150",
    "XGB30 + Cat150",
]:
    candidate_folds = ensemble_candidate_results[
        name
    ]["fold_results"].set_index("fold")

    fold_changes = pd.DataFrame({
        "original": original_folds["poisson_deviance"],
        "candidate": candidate_folds["poisson_deviance"],
    })

    fold_changes["change"] = (
        fold_changes["candidate"]
        - fold_changes["original"]
    )

    print(f"\n{name.upper()}")
    print("-" * 70)

    print(fold_changes.round(6).to_string())

    print(
        "Folds improved:",
        int((fold_changes["change"] < 0).sum()),
        "of 4"
    )

# Verify the reconstructed original ensemble
reconstructed_baseline = ensemble_candidate_results[
    "Original ensemble"
]["macro_metrics"]["poisson_deviance"]

assert np.isclose(
    reconstructed_baseline,
    ensemble_v1_results["macro_metrics"]["poisson_deviance"],
    atol=1e-6,
)

CATBOOST 150 — ENSEMBLE COMPARISON
        configuration  macro_deviance  pooled_deviance  macro_mae  macro_bias
Original XGB + Cat150        1.103669         1.093719   0.705683   -0.007296
       XGB30 + Cat150        1.104279         1.094198   0.703361    0.003605
    Original ensemble        1.106496         1.096307   0.706974   -0.004387

ORIGINAL XGB + CAT150
----------------------------------------------------------------------
        original  candidate    change
fold                                 
Fold 1  1.044600   1.042958 -0.001642
Fold 2  1.238840   1.232247 -0.006593
Fold 3  1.153896   1.155293  0.001398
Fold 4  0.988650   0.984179 -0.004471
Folds improved: 3 of 4

XGB30 + CAT150
----------------------------------------------------------------------
        original  candidate    change
fold                                 
Fold 1  1.044600   1.042933 -0.001667
Fold 2  1.238840   1.233582 -0.005258
Fold 3  1.153896   1.156928  0.003033
Fold 4  0.988650   0.983673 -0.

In [36]:
# Step 37: Diagnose the leading Version 2 ensemble candidate

diagnostic_df = ensemble_candidates[
    ["match_id", "fold", "y_true"]
].copy()

diagnostic_df["v1_prediction"] = (
    0.6 * ensemble_candidates["xgb_original"]
    + 0.4 * ensemble_candidates["cat_original"]
)

diagnostic_df["v2_prediction"] = (
    0.6 * ensemble_candidates["xgb_original"]
    + 0.4 * ensemble_candidates["cat_150"]
)

# Overall prediction behavior
prediction_summary = pd.DataFrame({
    "metric": [
        "Mean predicted goals",
        "Prediction standard deviation",
        "Minimum prediction",
        "Maximum prediction",
        "Mean absolute error",
    ],
    "original_ensemble": [
        diagnostic_df["v1_prediction"].mean(),
        diagnostic_df["v1_prediction"].std(),
        diagnostic_df["v1_prediction"].min(),
        diagnostic_df["v1_prediction"].max(),
        np.mean(
            np.abs(
                diagnostic_df["y_true"]
                - diagnostic_df["v1_prediction"]
            )
        ),
    ],
    "v2_candidate": [
        diagnostic_df["v2_prediction"].mean(),
        diagnostic_df["v2_prediction"].std(),
        diagnostic_df["v2_prediction"].min(),
        diagnostic_df["v2_prediction"].max(),
        np.mean(
            np.abs(
                diagnostic_df["y_true"]
                - diagnostic_df["v2_prediction"]
            )
        ),
    ],
})

print("PREDICTION BEHAVIOR — V1 VS. V2 CANDIDATE")
print("=" * 80)

print(
    prediction_summary.round(6).to_string(index=False)
)

# Compare prediction errors by actual goals scored
goal_count_rows = []

for goals, group in diagnostic_df.groupby("y_true"):

    goal_count_rows.append({
        "actual_goals": goals,
        "matches": len(group),
        "v1_mean_prediction": group["v1_prediction"].mean(),
        "v2_mean_prediction": group["v2_prediction"].mean(),
        "v1_mae": np.mean(
            np.abs(
                group["y_true"] - group["v1_prediction"]
            )
        ),
        "v2_mae": np.mean(
            np.abs(
                group["y_true"] - group["v2_prediction"]
            )
        ),
    })

goal_count_df = pd.DataFrame(
    goal_count_rows
).sort_values("actual_goals")

goal_count_df["mae_change"] = (
    goal_count_df["v2_mae"]
    - goal_count_df["v1_mae"]
)

print("\nERRORS BY ACTUAL GOALS SCORED")
print("=" * 100)

print(
    goal_count_df.round(6).to_string(index=False)
)

print(
    "\nTotal validation appearances:",
    len(diagnostic_df)
)

PREDICTION BEHAVIOR — V1 VS. V2 CANDIDATE
                       metric  original_ensemble  v2_candidate
         Mean predicted goals           0.766466      0.765868
Prediction standard deviation           0.065636      0.072295
           Minimum prediction           0.599467      0.592044
           Maximum prediction           0.889190      0.916793
          Mean absolute error           0.701696      0.699930

ERRORS BY ACTUAL GOALS SCORED
 actual_goals  matches  v1_mean_prediction  v2_mean_prediction   v1_mae   v2_mae  mae_change
            0      159            0.758412            0.755884 0.758412 0.755884   -0.002527
            1      121            0.774270            0.775063 0.225730 0.224937   -0.000793
            2       45            0.768601            0.769432 1.231399 1.230568   -0.000831
            3       13            0.794609            0.800566 2.205391 2.199434   -0.005956
            4        2            0.703597            0.697511 3.296403 3.302489    

In [38]:
# Step 38: Paired robustness analysis of the V2 candidate

from sklearn.metrics import mean_poisson_deviance

robustness_df = diagnostic_df.copy()

# Per-match Poisson deviance contributions
def poisson_deviance_contribution(y, prediction):
    prediction = np.clip(
        np.asarray(prediction, dtype=float),
        1e-9,
        None,
    )

    y = np.asarray(y, dtype=float)

    log_term = np.zeros_like(y, dtype=float)

    positive = y > 0

    log_term[positive] = (
        y[positive]
        * np.log(y[positive] / prediction[positive])
    )

    return 2 * (
        log_term - y + prediction
    )


robustness_df["v1_deviance"] = (
    poisson_deviance_contribution(
        robustness_df["y_true"],
        robustness_df["v1_prediction"],
    )
)

robustness_df["v2_deviance"] = (
    poisson_deviance_contribution(
        robustness_df["y_true"],
        robustness_df["v2_prediction"],
    )
)

robustness_df["deviance_change"] = (
    robustness_df["v2_deviance"]
    - robustness_df["v1_deviance"]
)

# Verify consistency with sklearn
assert np.isclose(
    robustness_df["v1_deviance"].mean(),
    mean_poisson_deviance(
        robustness_df["y_true"],
        robustness_df["v1_prediction"],
    ),
)

assert np.isclose(
    robustness_df["v2_deviance"].mean(),
    mean_poisson_deviance(
        robustness_df["y_true"],
        robustness_df["v2_prediction"],
    ),
)

print("PAIRED VALIDATION ANALYSIS")
print("=" * 75)

print(
    "Appearances improved:",
    int((robustness_df["deviance_change"] < 0).sum()),
)

print(
    "Appearances worsened:",
    int((robustness_df["deviance_change"] > 0).sum()),
)

print(
    "Mean per-match deviance change:",
    round(robustness_df["deviance_change"].mean(), 6),
)

print(
    "Median per-match deviance change:",
    round(robustness_df["deviance_change"].median(), 6),
)

# Paired, fold-stratified bootstrap
rng = np.random.default_rng(42)

n_bootstrap = 5000

fold_groups = [
    group["deviance_change"].to_numpy()
    for _, group in robustness_df.groupby("fold")
]

bootstrap_macro_changes = np.empty(n_bootstrap)

for iteration in range(n_bootstrap):

    sampled_fold_means = []

    for values in fold_groups:

        sampled = rng.choice(
            values,
            size=len(values),
            replace=True,
        )

        sampled_fold_means.append(sampled.mean())

    bootstrap_macro_changes[iteration] = np.mean(
        sampled_fold_means
    )

lower, upper = np.percentile(
    bootstrap_macro_changes,
    [2.5, 97.5],
)

print("\nFOLD-STRATIFIED PAIRED BOOTSTRAP")
print("=" * 75)

print(
    "Observed macro deviance change:",
    round(
        ensemble_candidate_results[
            "Original XGB + Cat150"
        ]["macro_metrics"]["poisson_deviance"]
        -
        ensemble_candidate_results[
            "Original ensemble"
        ]["macro_metrics"]["poisson_deviance"],
        6,
    ),
)

print(
    "Bootstrap 95% interval:",
    (round(lower, 6), round(upper, 6)),
)

print(
    "Bootstrap fraction favoring V2:",
    round(
        np.mean(bootstrap_macro_changes < 0),
        4,
    ),
)

PAIRED VALIDATION ANALYSIS
Appearances improved: 192
Appearances worsened: 148
Mean per-match deviance change: -0.002587
Median per-match deviance change: -0.001684

FOLD-STRATIFIED PAIRED BOOTSTRAP
Observed macro deviance change: -0.002827
Bootstrap 95% interval: (-0.006242, 0.000641)
Bootstrap fraction favoring V2: 0.9508


In [ ]:
# Step 39: Save the Version 2 experiment results

from datetime import datetime, timezone

V2_REPORT_DIR = PROJECT_ROOT / "experiments" / "v2" / "reports"

V2_REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Save experiment comparison tables
experiment_tables = {
    "xgboost_depth_results.csv": depth_results_df,
    "xgboost_regularization_results.csv": regularization_results_df,
    "xgboost_iterations_results.csv": iteration_results_df,
    "catboost_depth_results.csv": cat_depth_results_df,
    "catboost_iterations_results.csv": cat_iteration_results_df,
    "ensemble_comparison_results.csv": ensemble_candidate_df,
    "prediction_behavior_results.csv": prediction_summary,
    "errors_by_goal_count.csv": goal_count_df,
}

for filename, table in experiment_tables.items():

    output_path = V2_REPORT_DIR / filename

    table.to_csv(
        output_path,
        index=False,
    )

# Save the leading candidate's out-of-fold predictions
candidate_predictions = diagnostic_df[
    [
        "match_id",
        "fold",
        "y_true",
        "v1_prediction",
        "v2_prediction",
    ]
].copy()

candidate_predictions.to_csv(
    V2_REPORT_DIR / "v2_candidate_oof_predictions.csv",
    index=False,
)

# Save experiment metadata
experiment_summary = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": "experimental_not_promoted",
    "selection_metric": "macro_poisson_deviance",
    "validation_appearances": 340,
    "validation_folds": 4,
    "version_1": {
        "xgboost_iterations": 15,
        "catboost_iterations": 100,
        "xgboost_weight": 0.6,
        "catboost_weight": 0.4,
        "macro_poisson_deviance": float(
            ensemble_v1_results[
                "macro_metrics"
            ]["poisson_deviance"]
        ),
    },
    "version_2_candidate": {
        "xgboost_iterations": 15,
        "catboost_iterations": 150,
        "xgboost_weight": 0.6,
        "catboost_weight": 0.4,
        "macro_poisson_deviance": float(
            ensemble_candidate_results[
                "Original XGB + Cat150"
            ]["macro_metrics"]["poisson_deviance"]
        ),
    },
    "robustness": {
        "appearances_improved": int(
            (robustness_df["deviance_change"] < 0).sum()
        ),
        "appearances_worsened": int(
            (robustness_df["deviance_change"] > 0).sum()
        ),
        "bootstrap_iterations": n_bootstrap,
        "bootstrap_interval_95": [
            float(lower),
            float(upper),
        ],
        "bootstrap_fraction_favoring_v2": float(
            np.mean(bootstrap_macro_changes < 0)
        ),
    },
    "limitations": [
        "Candidate selected using the same validation folds.",
        "Bootstrap does not fully account for temporal dependence.",
        "No independent final-test evaluation performed.",
        "Version 1 models and artifacts remain unchanged.",
    ],
}

summary_path = V2_REPORT_DIR / "experiment_summary.json"

with open(summary_path, "w", encoding="utf-8") as file:
    json.dump(
        experiment_summary,
        file,
        indent=2,
    )

print("VERSION 2 EXPERIMENT RESULTS SAVED")
print("=" * 65)

for filename in experiment_tables:
    print(f"Saved: {filename}")

print("Saved: v2_candidate_oof_predictions.csv")
print("Saved: experiment_summary.json")

print("\nOutput directory:")
print(V2_REPORT_DIR)

print("\nVersion 1 artifacts remain unchanged.")

In [ ]:
# Step 40: Generate the Version 2 technical report

from datetime import datetime, timezone

report_path = V2_REPORT_DIR / "model_complexity_report.md"

v1_macro = ensemble_v1_results[
    "macro_metrics"
]["poisson_deviance"]

v2_macro = ensemble_candidate_results[
    "Original XGB + Cat150"
]["macro_metrics"]["poisson_deviance"]

relative_improvement = (
    (v1_macro - v2_macro) / v1_macro
) * 100

# Format experiment tables for Markdown
def markdown_table(df, columns=None):
    selected = df if columns is None else df[columns]

    headers = list(selected.columns)
    rows = selected.to_numpy().tolist()

    def format_value(value):
        if isinstance(value, (float, np.floating)):
            return f"{value:.6f}"
        return str(value)

    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]

    for row in rows:
        lines.append(
            "| "
            + " | ".join(format_value(value) for value in row)
            + " |"
        )

    return "\n".join(lines)


report = f"""# Version 2 — Model Complexity Experiments

## 1. Objective

I investigated whether controlled changes to XGBoost and CatBoost
model complexity could improve the goal-scoring predictions in my
Ronaldo 1,000-goal forecasting project.

I preserved the original Version 1 models, data contracts,
chronological validation folds, and final-test boundary.

All experiments were conducted separately under `experiments/v2/`.

## 2. Experimental Methodology

I evaluated candidate configurations using four expanding-window
chronological validation folds containing 340 validation appearances.

My primary selection metric was macro-averaged Poisson deviance.

I also examined pooled Poisson deviance, MAE, RMSE, prediction bias,
prediction variability, and fold-level stability.

I changed one hyperparameter at a time before evaluating selected
candidates within the original 60% XGBoost / 40% CatBoost ensemble.

The final-test period was not used for hyperparameter selection.

## 3. Original Baseline

| Component | Configuration |
|---|---|
| XGBoost | 15 iterations, depth 2 |
| CatBoost | 100 iterations, depth 2 |
| Ensemble | 60% XGBoost, 40% CatBoost |
| Macro Poisson deviance | {v1_macro:.6f} |

## 4. XGBoost Experiments

### 4.1 Tree Depth

{markdown_table(depth_results_df)}

Increasing tree depth did not improve overall macro Poisson deviance.
I retained depth 2.

### 4.2 Regularization

{markdown_table(regularization_results_df)}

Reducing L2 regularization from 20 to 10 produced a very small
improvement, but the effect was insufficient to justify changing
the baseline configuration.

### 4.3 Boosting Iterations

{markdown_table(iteration_results_df)}

Increasing XGBoost from 15 to 30 iterations improved individual-model
performance. However, this improvement did not carry over to the
original ensemble when CatBoost remained unchanged.

## 5. CatBoost Experiments

### 5.1 Tree Depth

{markdown_table(cat_depth_results_df)}

Deeper CatBoost trees substantially worsened macro Poisson deviance.
I retained depth 2.

### 5.2 Boosting Iterations

{markdown_table(cat_iteration_results_df)}

Increasing CatBoost from 100 to 150 iterations produced the strongest
individual CatBoost result among the configurations tested.

Performance deteriorated at 200 iterations.

## 6. Ensemble Comparison

{markdown_table(ensemble_candidate_df)}

The strongest ensemble combined the original 15-tree XGBoost model
with the 150-iteration CatBoost candidate.

The original 60/40 ensemble weights were preserved.

## 7. Leading Version 2 Candidate

| Component | Configuration |
|---|---|
| XGBoost | Original 15 iterations |
| CatBoost | 150 iterations |
| Ensemble weights | 60% XGBoost / 40% CatBoost |
| Macro Poisson deviance | {v2_macro:.6f} |
| Relative macro improvement | {relative_improvement:.4f}% |

The candidate improved macro Poisson deviance in three of four
chronological validation folds.

## 8. Prediction Behavior

{markdown_table(prediction_summary)}

The updated ensemble produced slightly more variable predictions,
while its average predicted goals remained close to the original.

The improvement was not driven by a large shift in overall
prediction magnitude.

## 9. Robustness Assessment

I compared the original and updated ensembles on the same
340 validation appearances.

- Appearances improved: 192
- Appearances worsened: 148
- Observed macro deviance change: {v2_macro - v1_macro:.6f}
- Bootstrap iterations: {n_bootstrap}
- Exploratory bootstrap 95% interval:
  ({lower:.6f}, {upper:.6f})
- Bootstrap fraction favoring Version 2:
  {np.mean(bootstrap_macro_changes < 0):.4f}

The bootstrap resampled appearances within each validation fold.

The interval includes zero, so the analysis does not establish
a reliable generalization improvement.

The bootstrap also does not fully account for temporal dependence.

## 10. Limitations

1. I tested multiple configurations on the same validation folds,
   introducing model-selection uncertainty.
2. The observed improvement is modest.
3. The bootstrap is exploratory and does not fully capture
   temporal dependence between appearances.
4. I have not performed an independent final-test evaluation
   of the Version 2 candidate.
5. Improvements in goal-count prediction do not automatically
   establish improvements in milestone-date forecasting.

## 11. Conclusion

I identified a promising experimental configuration by increasing
CatBoost iterations from 100 to 150 while preserving the original
XGBoost model and ensemble weights.

The candidate reduced macro Poisson deviance from
{v1_macro:.6f} to {v2_macro:.6f}, an improvement of
approximately {relative_improvement:.2f}%.

However, the uncertainty analysis and repeated use of the same
validation folds prevent me from treating this as a confirmed
generalization improvement.

I therefore retain this configuration as an experimental
Version 2 candidate.

Version 1 remains frozen and unchanged.

## 12. Reproducibility

Notebook:
`experiments/v2/notebooks/01_model_complexity.ipynb`

Experiment outputs:
`experiments/v2/reports/`

Generated:
{datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")}
"""

report_path.write_text(
    report,
    encoding="utf-8",
)

print("VERSION 2 TECHNICAL REPORT")
print("=" * 65)
print("Saved:", report_path)
print("Characters:", len(report))
print("Sections:", report.count("\n## "))
print("\nVersion 1 remains unchanged.")

In [44]:
# Step 41: Verify the Version 2 experiment deliverables

import json

required_files = [
    "xgboost_depth_results.csv",
    "xgboost_regularization_results.csv",
    "xgboost_iterations_results.csv",
    "catboost_depth_results.csv",
    "catboost_iterations_results.csv",
    "ensemble_comparison_results.csv",
    "prediction_behavior_results.csv",
    "errors_by_goal_count.csv",
    "v2_candidate_oof_predictions.csv",
    "experiment_summary.json",
    "model_complexity_report.md",
]

print("VERSION 2 — FINAL VERIFICATION")
print("=" * 70)

# Check that all expected files exist
missing_files = [
    filename
    for filename in required_files
    if not (V2_REPORT_DIR / filename).is_file()
]

assert not missing_files, (
    f"Missing files: {missing_files}"
)

print(f"Files verified: {len(required_files)}/{len(required_files)}")

# Verify saved predictions
saved_predictions = pd.read_csv(
    V2_REPORT_DIR / "v2_candidate_oof_predictions.csv"
)

assert len(saved_predictions) == 340
assert saved_predictions["match_id"].is_unique
assert saved_predictions["fold"].nunique() == 4

print("OOF predictions: 340 unique appearances")
print("Chronological folds: 4")

# Verify experiment metadata
with open(
    V2_REPORT_DIR / "experiment_summary.json",
    encoding="utf-8",
) as file:
    saved_summary = json.load(file)

assert saved_summary["status"] == "experimental_not_promoted"

assert np.isclose(
    saved_summary["version_1"]["macro_poisson_deviance"],
    1.106496,
    atol=1e-6,
)

assert np.isclose(
    saved_summary["version_2_candidate"]["macro_poisson_deviance"],
    1.103669,
    atol=1e-6,
)

print("Experiment metadata: verified")

# Verify report structure and key metrics
report_text = (
    V2_REPORT_DIR / "model_complexity_report.md"
).read_text(encoding="utf-8")

required_sections = [
    "## 1. Objective",
    "## 2. Experimental Methodology",
    "## 6. Ensemble Comparison",
    "## 9. Robustness Assessment",
    "## 10. Limitations",
    "## 11. Conclusion",
    "## 12. Reproducibility",
]

assert all(
    section in report_text
    for section in required_sections
)

assert "1.106496" in report_text
assert "1.103669" in report_text

print("Technical report: verified")

print("\n" + "=" * 70)
print("ALL VERSION 2 EXPERIMENT CHECKS PASSED")
print("Version 1 remains unchanged.")

VERSION 2 — FINAL VERIFICATION
Files verified: 11/11
OOF predictions: 340 unique appearances
Chronological folds: 4
Experiment metadata: verified
Technical report: verified

ALL VERSION 2 EXPERIMENT CHECKS PASSED
Version 1 remains unchanged.
